In [2]:
import torch
import math

In [122]:
def splitter_tree_powers(
    n: int,
    sigma: float,
    *,
    batch: int = 1,
    eps: float = 1e-6,
    device=None,
    dtype=torch.float64,
    generator: torch.Generator | None = None,
) -> torch.Tensor:
    """
    Balanced splitter tree (Gaussian i.i.d. ratios) that:
      • Handles non-powers of two by building to the next power-of-two (m)
        and center-cropping the m leaves down to n.
      • Supports an arbitrary batch dimension.
      • Returns a tensor of shape (batch, n) whose rows sum to n.
    """
    if n < 1:
        raise ValueError("n must be ≥ 1")

    m = 1 << (n - 1).bit_length()  # smallest 2^k ≥ n
    levels = int(math.log2(m))

    powers = (
        0.5 * m * torch.ones((batch, 1), device=device, dtype=dtype)
    )  # start with 1 W

    for _ in range(levels):
        k = powers.size(1)  # nodes at this level
        ratios = torch.normal(
            0.5, sigma, size=(batch, k), device=device, dtype=dtype, generator=generator
        ).clamp(eps, 1.0 - eps)

        left = ratios * powers
        right = (1.0 - ratios) * powers

        # Interleave: L1,R1,L2,R2,…  — works for any batch size, including 1
        new_powers = torch.empty((batch, k * 2), device=device, dtype=dtype)
        new_powers[:, 0::2] = left
        new_powers[:, 1::2] = right
        powers = new_powers  # (batch, 2k)

    # Center-crop from m leaves down to n leaves
    if n < m:
        start = (m - n) // 2
        powers = powers[:, start : start + n]

    # Renormalize so each row sums exactly to n

    return powers


output = splitter_tree_powers(n=8, sigma=0.01, batch=2)
output

tensor([[0.4798, 0.4887, 0.5035, 0.5029, 0.4881, 0.5160, 0.5122, 0.5088],
        [0.4543, 0.4855, 0.5054, 0.5037, 0.5171, 0.5138, 0.5199, 0.5003]],
       dtype=torch.float64)

In [104]:
torch.finfo(torch.float32).eps

1.1920928955078125e-07

In [105]:
mrm_out = torch.clamp(torch.abs(torch.randn(3, 6)), min=1e-6, max=1.0)
# apply splitter tree output to mrm_out as a piecewise multiplication
splitter_tree_output = splitter_tree_powers(n=6, sigma=0.03, batch=3)
mrm_out_ler = torch.mul(mrm_out, splitter_tree_output)
print(mrm_out)
print(splitter_tree_output)
print(mrm_out_ler)

tensor([[0.3436, 1.0000, 1.0000, 1.0000, 1.0000, 0.5775],
        [0.1186, 0.6762, 0.0681, 0.4546, 1.0000, 0.4949],
        [0.1550, 0.5062, 0.9108, 0.2656, 0.8704, 0.8416]])
tensor([[0.9718, 1.0442, 1.0521, 0.9259, 1.1232, 0.9622],
        [1.1227, 0.8865, 1.1556, 0.8698, 0.9118, 1.0218],
        [1.1035, 1.0761, 1.1356, 0.9086, 0.8713, 0.9021]], dtype=torch.float64)
tensor([[0.3339, 1.0442, 1.0521, 0.9259, 1.1232, 0.5557],
        [0.1332, 0.5995, 0.0787, 0.3954, 0.9118, 0.5057],
        [0.1711, 0.5448, 1.0343, 0.2413, 0.7583, 0.7591]], dtype=torch.float64)
